# Gene2Wire BARseq — OnDemand 0908

        This notebook runs the shared, versioned paper experiment code. It contains no
        dataset-specific model patches. Use a Python 3.10+ OnDemand kernel with the repository's
        experiment dependencies available; the notebook does not install packages into the kernel.

        Run cells from top to bottom. The first uncached use downloads the pinned code and raw
        data; subsequent runs reuse verified local files. Checkpoints persist across kernel
        disconnects. All result tables and predictions go beneath
        `/home/yueyue/gene2wire/paper_figure_exports`; plots display here and save as PDF only.

        Outputs are deliberately cleared. Earlier paper numbers remain provisional until
        the harmonized reruns and their diagnostics have been reviewed.


In [ ]:
from pathlib import Path
import os

# All notebooks share these defaults. Change switches here before Run All.
N_OUTER_FOLDS = 3
USE_LOCATION = False
USE_TARGET_FEATURES = False
N_JOBS = 32
N_REPETITIONS = 5
STRATEGY = 'full_joint'
SEED = 20260908

RUN_INFORMATION_CONTROLS = True
RUN_RANDOM_FOREST = True
RUN_MECHANISM_CONTROLS = True
RUN_CALIBRATION_CONTROLS = True
RUN_QIAO = False

BASE_DIR = Path('/home/yueyue/gene2wire').expanduser()
RAW_DATA_DIR = BASE_DIR / 'raw_data'
CHECKPOINT_DIR = BASE_DIR / 'checkpoints' / '0908'
EXPORT_DIR = BASE_DIR / 'paper_figure_exports'
FIGURE_DIR = BASE_DIR / 'figures' / '0908'
CODE_CACHE_DIR = BASE_DIR / 'code'

CORE_COMMIT = '2ad6a826a0e28dc8ff2dec3540003d9660b27531'
EXPECTED_SOURCE_HASH = '64180d34d742aa71e01ab29962d565097f9d73be636ae0276acdaa57a79246d6'
REPO_URL = 'https://github.com/Yue-stat/Gene2Wire.git'

# Set before importing NumPy/SciPy. Parallelism is at fold/repetition level.
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '1'

REQUIRED_MODULES = ['numpy', 'scipy', 'pandas', 'sklearn', 'joblib', 'matplotlib', 'yaml', 'IPython']


**Load the pinned code.** A verified local checkout works offline.
        A stale package already imported in this kernel requires a kernel restart;
        the notebook never reloads or rewrites installed model source.


In [ ]:
import hashlib
import importlib.util
import re
import shutil
import subprocess
import sys
import tempfile

if sys.version_info < (3, 10):
    raise RuntimeError('Select an OnDemand Python 3.10 or newer kernel.')
if not re.fullmatch(r'[0-9a-f]{40}', CORE_COMMIT):
    raise RuntimeError('This notebook needs its released 40-character CORE_COMMIT pin.')
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_SOURCE_HASH):
    raise RuntimeError('This notebook needs its released source checksum.')

def notebook_source_hash(package_root):
    # Same byte-level convention as experiments.protocol.source_hash().
    digest = hashlib.sha256()
    for source_path in sorted(package_root.rglob('*.py')):
        digest.update(source_path.relative_to(package_root).as_posix().encode())
        digest.update(source_path.read_bytes())
    return digest.hexdigest()

def verify_checkout(checkout, require_git_pin=True):
    package_root = checkout / 'src' / 'gene2wire'
    if not package_root.is_dir():
        raise RuntimeError(f'Missing Gene2Wire sources in {checkout}')
    if require_git_pin:
        actual_commit = subprocess.check_output(
            ['git', '-C', str(checkout), 'rev-parse', 'HEAD'], text=True).strip()
        if actual_commit != CORE_COMMIT:
            raise RuntimeError(f'Cached code has commit {actual_commit}, expected {CORE_COMMIT}.')
    if notebook_source_hash(package_root) != EXPECTED_SOURCE_HASH:
        raise RuntimeError(f'Source checksum mismatch in {checkout}; use the released code.')
    return checkout

# Running from the exact local repository is supported without any network call.
CORE_CHECKOUT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    package_root = candidate / 'src' / 'gene2wire'
    if package_root.is_dir() and notebook_source_hash(package_root) == EXPECTED_SOURCE_HASH:
        CORE_CHECKOUT = verify_checkout(candidate, require_git_pin=False)
        break

if CORE_CHECKOUT is None:
    CODE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cached_checkout = CODE_CACHE_DIR / CORE_COMMIT
    if not cached_checkout.exists():
        stage = Path(tempfile.mkdtemp(prefix='.gene2wire-download-', dir=CODE_CACHE_DIR))
        try:
            for arguments in (
                ['git', 'init', '--quiet', str(stage)],
                ['git', '-C', str(stage), 'remote', 'add', 'origin', REPO_URL],
                ['git', '-C', str(stage), 'fetch', '--quiet', '--depth', '1', 'origin', CORE_COMMIT],
                ['git', '-C', str(stage), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'],
            ):
                subprocess.run(arguments, check=True)
            verify_checkout(stage)
            try:
                stage.rename(cached_checkout)
            except OSError:
                # Another notebook may have completed this same immutable cache.
                if not cached_checkout.exists():
                    raise
                verify_checkout(cached_checkout)
        finally:
            if stage.exists():
                shutil.rmtree(stage)
    CORE_CHECKOUT = verify_checkout(cached_checkout)

existing = sys.modules.get('gene2wire')
if existing is not None:
    same_path = Path(existing.__file__).resolve().parent == (CORE_CHECKOUT / 'src' / 'gene2wire').resolve()
    same_source = getattr(existing, '_notebook_source_hash_0908', None) == EXPECTED_SOURCE_HASH
    if not (same_path and same_source):
        raise RuntimeError('A different or unverified gene2wire is already imported. Restart the kernel, then Run All.')

missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Use an OnDemand Python kernel containing these dependencies: '
                       + ', '.join(missing) + '. See the repository environment instructions.')
sys.path.insert(0, str(CORE_CHECKOUT / 'src'))
import gene2wire
from gene2wire.experiments.protocol import Settings, source_hash
if source_hash() != EXPECTED_SOURCE_HASH:
    raise RuntimeError('Imported code does not match the released source checksum.')
gene2wire._notebook_source_hash_0908 = EXPECTED_SOURCE_HASH
for directory in (RAW_DATA_DIR, CHECKPOINT_DIR, EXPORT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'core_commit': CORE_COMMIT, 'source_hash': source_hash(),
       'imported_from': gene2wire.__file__, 'raw_cache': str(RAW_DATA_DIR),
       'checkpoints': str(CHECKPOINT_DIR), 'exports': str(EXPORT_DIR)})


**Shared scientific settings.** `full_joint` evaluates simultaneous
        rank/penalty candidates from a deterministic bounded Cartesian grid. Exact direct and
        residual-off endpoints are included in Joint's budget. Inner validation selects models;
        final preprocessing, calibration, and fitting use the designated development data.
        Mechanism/calibration stress tests are simulation controls. Every model shares each
        scenario's observation mask and paired-reference subset.


In [ ]:
from dataclasses import asdict
import numpy as np
import pandas as pd
from IPython.display import display
from gene2wire.experiments.pipeline import run_experiment, run_simulation_experiments
from gene2wire.experiments.plotting import plot_results
from gene2wire.tuning import full_joint_candidates

settings = Settings(
    n_outer_folds=N_OUTER_FOLDS, use_location=USE_LOCATION,
    use_target_features=USE_TARGET_FEATURES, n_jobs=N_JOBS,
    n_repetitions=N_REPETITIONS, strategy=STRATEGY, seed=SEED,
    run_information_controls=RUN_INFORMATION_CONTROLS,
    run_random_forest=RUN_RANDOM_FOREST,
    run_mechanism_controls=RUN_MECHANISM_CONTROLS,
    run_calibration_controls=RUN_CALIBRATION_CONTROLS,
    run_qiao=RUN_QIAO,
)
display(pd.DataFrame([asdict(settings)]).T.rename(columns={0: 'setting'}))


In [ ]:
def preflight(dataset):
    dataset.validate()
    folds = tuple(dataset.split_builder(settings.n_outer_folds, settings.seed))
    for fold in folds:
        fold.validate(len(dataset.cell_ids))
    features = dataset.feature_builder(
        folds[0].train_rows, settings.use_location, settings.use_target_features)
    roles = []
    for fold in folds:
        roles.append({'dataset': dataset.name, 'outer_fold': fold.outer_fold,
                      'inner_train_cells': len(fold.train_rows),
                      'validation_cells': len(fold.validation_rows),
                      'test_cells': len(fold.test_rows),
                      'split_design': dict(fold.metadata)})
    scalar_metadata = {key: value for key, value in dataset.metadata.items()
                       if value is None or isinstance(value, (str, bool, int, float))}
    display(pd.DataFrame([{'dataset': dataset.name, 'cells': len(dataset.cell_ids),
                           'targets': len(dataset.target_ids),
                           'measured_pairs': int(dataset.measured.sum()),
                           'reference_positives': int(dataset.reference.sum()),
                           'feature_columns': features.X.shape[1],
                           'feature_blocks': dict(features.feature_blocks),
                           'target_feature_columns': 0 if features.Y_target is None else features.Y_target.shape[1],
                           'natural_paired_assay': dataset.natural_observed is not None}]))
    display(pd.DataFrame(roles))
    display(pd.DataFrame([scalar_metadata]).T.rename(columns={0: 'dataset metadata'}))
    tuning = settings.tuning_config(features.X.shape[1], len(dataset.target_ids))
    display(pd.DataFrame([
        {'model': model.name, 'strategy': settings.strategy,
         'maximum_trials': tuning.candidate_budget,
         'bounded_grid_candidates': len(full_joint_candidates(model, tuning)),
         'eligible_structures': sorted({c.kind for c in full_joint_candidates(model, tuning)})}
        for model in settings.models()
    ]))
    print('Features above are fitted on inner-training cells only. Final refit uses the development cells.')
    print('Repeated masks share a biological dataset; they are not additional independent animals.')
    return folds


**BARseq A1 and M1.** Both panels are fitted and reported separately.
            Location uses the native measured spatial covariates. Optional target features are
            anatomy descriptors derived from target names, not postsynaptic gene expression.
            Qiao comparisons are reserved for the simulation's declared target descriptors.


In [ ]:
from gene2wire.experiments.datasets.barseq import load_barseq
if RUN_QIAO:
    raise ValueError('The Qiao comparison is enabled in simulation only; use RUN_QIAO=False here.')
datasets = load_barseq(RAW_DATA_DIR / 'BARseq')
PANELS = ('A1', 'M1')
for panel in PANELS:
    preflight(datasets[panel])


In [ ]:
all_artifacts = {}
for panel in PANELS:
    all_artifacts[panel] = run_experiment(
        dataset=datasets[panel], settings=settings,
        checkpoint_dir=CHECKPOINT_DIR, export_dir=EXPORT_DIR)


**Loss-rate curves and PDF figures.** Curves retain the complete configured
        loss-rate grid. Simulation includes all three sharing strengths; BARseq includes both panels.
        The natural Projection-TAGs analysis uses paired-audit and model-comparison panels.
        No PNG files are written.


In [ ]:
for label, artifacts in all_artifacts.items():
    plot_results(artifacts, output_dir=FIGURE_DIR)


**Result tables and reusable exports.**


In [ ]:
# Full tables and predictions are already exported by the shared pipeline.
# The notebook presents one wide metric summary; every detailed table remains on disk.
for label, artifacts in all_artifacts.items():
    print(label)
    candidate_names = ('aggregate', 'summary', 'summary_metrics', 'metrics_summary', 'aggregate_metrics', 'metrics')
    chosen = next((artifacts.tables[name] for name in candidate_names
                   if name in artifacts.tables), None)
    if chosen is None:
        chosen = next((value for value in artifacts.tables.values()
                       if isinstance(value, pd.DataFrame)), None)
    if chosen is not None:
        display(chosen)
    print('Full result export:', artifacts.export_dir)
    print('Saved table names:', list(artifacts.tables))
print('Figure PDFs:', FIGURE_DIR)
print('Reusable raw cache:', RAW_DATA_DIR)
print('Resumable checkpoints:', CHECKPOINT_DIR)
